# Feature engineering (Dataset B)
Week 6: add features to the best tree model from 04 and show which ones help, using rolling time-based cross-validation. CRISP-DM Data Preparation, task 3.3 Construct data: every derived attribute below says why it was made.

Deliverable (week-by-week doc): a table comparing old vs new feature sets, including the school district layer.

Dataset A is unchanged. Extra raw columns are joined here on `ListingId`, so the team's shared CSVs and 03 stay as they are. The feature functions live in `idx_pipeline.py`.

#### Setup

In [1]:
import os
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline

from idx_pipeline import (CAT_COLS, DATA_DIR, SEED, TARGET, DatasetAImputer, add_basic_features, add_comps,
                          add_extras, add_fee_features, add_geo_features, add_market_trend, build_comp_pool,
                          build_preprocessor, fill_coords, load_extras, load_split, metrics, rolling_folds)

print("sklearn", sklearn.__version__)
np.random.seed(SEED)
MODEL_DIR = os.path.join(DATA_DIR, "models")

sklearn 1.9.1


#### Test design (written before training)
- **Rolling time-based cross-validation** on Nov 2025 – Mar 2026: each fold trains on every month before its validation month, then scores that month. Fold 5 is the same split as 03 and 04.
- **Why not random or stratified folds**: they would train on future sales to predict past ones, which makes scores look better than real use. Price bands are still reported.
- **Fit inside each fold**: fills, encoders, the comps pool, KMeans and market medians are rebuilt from that fold's train rows only.
- **Test is not used.** The raw April 2026 file isn't read.
- **Test model**: `rf_leaf5_log` from 04, with 100 trees instead of 200 so that 45+ fits are practical. Scores are for comparing feature sets, so they can differ slightly from 04.
- **Noise check**: a paired bootstrap of all validation homes across folds gives a 95% interval on the change. A group counts as helping only if the whole MdAPE interval is below 0 **and** it improves MdAPE in at least 4 of the 5 folds.
- **Known limit**: the ClosePrice outlier limits in Dataset A come from all of train, so earlier folds' rows were filtered with limits that saw later months. This doesn't touch the features or models, and the effect should be small.

#### Step 1: Load and build every feature

In [2]:
VAL_MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03"]
dev = pd.concat([load_split(s) for s in ["train", "val"]], ignore_index=True)  # test stays out
extras = load_extras()  # raw columns that Dataset A leaves out, read up to Mar 2026


def engineer(df, lat, lon, pool, kmeans, train):
    out = add_fee_features(add_basic_features(add_extras(df, extras)))
    out = add_geo_features(out, lat, lon)
    out["geo_cluster"] = kmeans.predict(np.column_stack([lat, lon])).astype(str)
    out = add_comps(out, lat, lon, pool)
    return add_market_trend(out, train)


folds = {}
start = time.time()
for month, train, val in rolling_folds(dev, VAL_MONTHS):
    # everything learned here comes from this fold's train rows only
    imputer = DatasetAImputer().fit(train)
    lat, lon = fill_coords(train, imputer)
    pool = build_comp_pool(train, lat, lon)
    kmeans = MiniBatchKMeans(n_clusters=100, n_init=3, random_state=SEED).fit(np.column_stack([lat, lon]))
    folds[month] = (engineer(train, lat, lon, pool, kmeans, train),
                    engineer(val, *fill_coords(val, imputer), pool, kmeans, train))
    assert folds[month][0]["source_month"].max() < month  # train is always earlier than validation
    print(month, "train", len(train), "val", len(val), f"{time.time() - start:.0f}s")

y_all = np.concatenate([folds[m][1][TARGET].to_numpy() for m in VAL_MONTHS])  # every validation home, in order

2025-11 train 246137 val 9596 34s


2025-12 train 255733 val 10317 73s


2026-01 train 266050 val 7370 112s


2026-02 train 273420 val 8859 154s


2026-03 train 282279 val 11412 209s


In [3]:
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()
BASE_NUM = [c for c in features if c not in CAT_COLS]  # Dataset A, same as 04

# group: (numeric columns, categorical columns), with the idea behind each
GROUPS = {
    # Team and week-by-week doc ideas: ratios, yard size, combined columns, and flooring as a finish-quality signal
    "columns": (["bed_bath_ratio", "sqft_per_bed", "baths_per_bed", "stories_filled", "yard_sqft", "lot_coverage",
                 "main_level_share", "has_wood", "has_stone", "has_tile", "has_carpet", "has_laminate", "has_vinyl",
                 "Flooring_missing", "in_subdivision", "flag_small_for_beds", "flag_many_baths"],
                ["MLSAreaMajor", "SubdivisionName"]),
    # On hold from 02 until today: HOA fee and whether it's known
    "hoa": (["fee_monthly", "HasHOA", "AssociationFee_missing"], []),
    # Location was the biggest signal in 04: diagonals, commute distance, coast, and neighborhoods finer than ZIP
    "geo": (["rot_15", "rot_30", "rot_45", "rot_60", "km_la", "km_sf", "km_sj", "km_sd", "km_sac", "km_job_center",
             "km_coast"], ["geo_cluster"]),
    # STAND-IN for the required school district layer, until the spatial join (needs the school district shapefile)
    "school_raw": ([], ["HighSchoolDistrict"]),
    # What nearby homes sold for in the previous 6 months, aimed at the weak top price band
    "comps": (["comp_ppsf_median", "comp_ppsf_mean", "comp_km_kth", "comp_count", "sales_within_1km",
               "comp_implied_value"], []),
    # month_sin/cos only capture season. These capture whether prices are rising or falling
    "market": (["market_zip_3m", "market_county_3m", "market_county_12m", "market_county_change"], []),
}
for name, (num, cat) in GROUPS.items():
    print(f"{name}: {len(num)} numeric, {len(cat)} categorical")

columns: 17 numeric, 2 categorical
hoa: 3 numeric, 0 categorical
geo: 11 numeric, 1 categorical
school_raw: 0 numeric, 1 categorical
comps: 6 numeric, 0 categorical
market: 4 numeric, 0 categorical


In [4]:
rows, apes, keep_models = {}, {}, {}


def make_model(kind, num, cat):
    pipe = Pipeline([("pre", build_preprocessor(kind, num, cat)),
                     ("model", RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=SEED))])
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)


def run(name, extra_num=(), extra_cat=(), kind="tree_te", cats=CAT_COLS):
    # fit once per fold, on that fold's train rows only
    num, cat = BASE_NUM + list(extra_num), list(cats) + list(extra_cat)
    cols = list(dict.fromkeys(num + CAT_COLS + cat))
    start, preds, per_fold = time.time(), [], {}
    for month in VAL_MONTHS:
        train, val = folds[month]
        model = make_model(kind, num, cat).fit(train[cols], train[TARGET].to_numpy())
        preds.append(model.predict(val[cols]))
        per_fold[f"MdAPE_{month}"] = metrics(val[TARGET].to_numpy(), preds[-1])["MdAPE_%"]
    # overfit gap on the last fold only, because predicting on train in every fold would double the time
    train_score = metrics(train[TARGET].to_numpy(), model.predict(train[cols]))["MdAPE_%"]
    pred = np.concatenate(preds)
    pooled = metrics(y_all, pred)
    rows[name] = {**{f"cv_{k}": v for k, v in pooled.items()}, **per_fold,
                  "gap_last_fold": round(per_fold[f"MdAPE_{VAL_MONTHS[-1]}"] - train_score, 2),
                  "n_inputs": len(num) + len(cat), "fit_s": round(time.time() - start)}
    apes[name] = np.abs(pred - y_all) / y_all
    keep_models[name] = (kind, num, cat, cols)  # the recipe, not the forests, to save memory
    print(name, rows[name]["cv_MdAPE_%"], rows[name]["cv_MAPE_%"])


def bootstrap_change(new, ref, n=1000):
    # paired: the same resampled homes for both models. Changes are in points, and negative means the new model is better.
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y_all), (n, len(y_all)))
    a, b = apes[new][idx], apes[ref][idx]
    med = 100 * (np.median(a, axis=1) - np.median(b, axis=1))
    mean = 100 * (a.mean(axis=1) - b.mean(axis=1))
    wins = sum(rows[new][f"MdAPE_{m}"] < rows[ref][f"MdAPE_{m}"] for m in VAL_MONTHS)
    return {"MdAPE_change": round(100 * (np.median(apes[new]) - np.median(apes[ref])), 2),
            "ci_low": round(np.percentile(med, 2.5), 2), "ci_high": round(np.percentile(med, 97.5), 2),
            "folds_better": wins,
            "MAPE_change": round(100 * (apes[new].mean() - apes[ref].mean()), 2),
            "mape_ci_low": round(np.percentile(mean, 2.5), 2), "mape_ci_high": round(np.percentile(mean, 97.5), 2)}

#### Step 2: How to encode City, county and ZIP
Integer codes (what 04 used) vs target encoding (team decision, cross-fitted so a home's encoding never sees its own price), and ZIP alone, since a VIF check found that these location columns overlap.

In [5]:
run("base_ordinal", kind="tree")
run("target_encoded")
run("target_encoded_zip_only", cats=["PostalCode"])
pd.DataFrame(rows).T

base_ordinal 7.78 11.65


target_encoded 7.85 11.68


target_encoded_zip_only 7.91 11.86


,cv_MdAPE_%,cv_MAPE_%,cv_R2,cv_MAE_$,cv_within_10%,MdAPE_2025-11,MdAPE_2025-12,MdAPE_2026-01,MdAPE_2026-02,MdAPE_2026-03,gap_last_fold,n_inputs,fit_s
base_ordinal,7.78,11.65,0.8838,152210.0,59.8,7.72,7.70,7.82,7.98,7.75,2.99,34.0,209.0
target_encoded,7.85,11.68,0.8814,153871.0,59.6,7.87,7.82,7.74,8.03,7.78,2.67,34.0,246.0
target_encoded_zip_only,7.91,11.86,0.8781,156129.0,59.2,7.89,7.92,7.89,8.09,7.79,2.73,32.0,217.0


#### Step 3: Add one feature group at a time
Each group is added to `target_encoded`, so all the new categoricals can be target-encoded too.

In [6]:
for name, (num, cat) in GROUPS.items():
    run(f"+{name}", num, cat)

ablation = pd.DataFrame({name: bootstrap_change(f"+{name}", "target_encoded") for name in GROUPS}).T
ablation["helps"] = (ablation["ci_high"] < 0) & (ablation["folds_better"] >= 4)
ablation.sort_values("MdAPE_change")

+columns 7.76 11.55


+hoa 7.77 11.58


+geo 7.59 11.38


+school_raw 7.83 11.69


+comps 8.0 11.75


+market 7.94 11.9


,MdAPE_change,ci_low,ci_high,folds_better,MAPE_change,mape_ci_low,mape_ci_high,helps
geo,-0.26,-0.32,-0.21,5.0,-0.29,-0.33,-0.26,True
columns,-0.09,-0.15,-0.03,5.0,-0.12,-0.16,-0.08,True
hoa,-0.08,-0.12,-0.04,5.0,-0.09,-0.11,-0.07,True
school_raw,-0.02,-0.06,0.02,3.0,0.02,0.00,0.03,False
market,0.09,0.02,0.17,0.0,0.22,0.18,0.27,False
comps,0.15,0.07,0.23,0.0,0.08,0.01,0.14,False


#### Step 4: Combine the groups that helped, then everything

In [7]:
helped = ablation.index[ablation["helps"]].tolist()
print("groups that helped:", helped)
if helped:
    run("kept_groups", [c for g in helped for c in GROUPS[g][0]], [c for g in helped for c in GROUPS[g][1]])

run("all_groups", [c for g in GROUPS.values() for c in g[0]], [c for g in GROUPS.values() for c in g[1]])
pd.DataFrame({n: bootstrap_change(n, "target_encoded") for n in ["kept_groups", "all_groups"] if n in rows}).T

groups that helped: ['columns', 'hoa', 'geo']


kept_groups 7.5 11.22


all_groups 7.77 11.44


,MdAPE_change,ci_low,ci_high,folds_better,MAPE_change,mape_ci_low,mape_ci_high
kept_groups,-0.35,-0.42,-0.29,5.0,-0.46,-0.50,-0.42
all_groups,-0.08,-0.17,-0.00,3.0,-0.23,-0.29,-0.17


#### Step 5: Old vs new feature sets (week 6 deliverable)
Cross-validated across all 5 validation months, with the same homes on every row. `base_ordinal` is the old feature set. `school_raw` is a stand-in: the required spatial join needs the school district shapefile.

In [8]:
results = pd.DataFrame(rows).T
assert all(len(a) == len(y_all) for a in apes.values())  # every row scored on the same homes
results.to_csv(os.path.join(MODEL_DIR, "fe_cv_metrics.csv"))
ablation.to_csv(os.path.join(MODEL_DIR, "fe_cv_ablation.csv"))
results.sort_values("cv_MdAPE_%")

,cv_MdAPE_%,cv_MAPE_%,cv_R2,cv_MAE_$,cv_within_10%,MdAPE_2025-11,MdAPE_2025-12,MdAPE_2026-01,MdAPE_2026-02,MdAPE_2026-03,gap_last_fold,n_inputs,fit_s
kept_groups,7.50,11.22,0.8911,147016.0,61.2,7.47,7.44,7.54,7.65,7.46,2.87,68.0,731.0
+geo,7.59,11.38,0.8885,149199.0,60.5,7.45,7.52,7.52,7.79,7.65,2.87,46.0,482.0
+columns,7.76,11.55,0.8842,151704.0,59.7,7.80,7.67,7.70,8.00,7.70,2.81,53.0,395.0
+hoa,7.77,11.58,0.8830,152659.0,59.8,7.73,7.71,7.73,8.00,7.70,2.66,37.0,236.0
all_groups,7.77,11.44,0.8943,148456.0,59.8,7.62,7.60,7.62,8.17,7.79,3.34,79.0,999.0
base_ordinal,7.78,11.65,0.8838,152210.0,59.8,7.72,7.70,7.82,7.98,7.75,2.99,34.0,209.0
+school_raw,7.83,11.69,0.8815,153943.0,59.4,7.81,7.76,7.78,8.03,7.77,2.64,35.0,215.0
target_encoded,7.85,11.68,0.8814,153871.0,59.6,7.87,7.82,7.74,8.03,7.78,2.67,34.0,246.0
target_encoded_zip_only,7.91,11.86,0.8781,156129.0,59.2,7.89,7.92,7.89,8.09,7.79,2.73,32.0,217.0
+market,7.94,11.90,0.8773,156968.0,58.8,7.90,7.84,7.83,8.14,7.99,3.10,38.0,313.0


#### Step 6: Where it helped: price bands

In [9]:
best = results["cv_MdAPE_%"].astype(float).idxmin()
print("best feature set:", best)
bands = pd.qcut(pd.Series(y_all), 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
by_band = {}
for name in dict.fromkeys(["base_ordinal", best]):
    e = pd.Series(apes[name])
    by_band[name] = pd.DataFrame({"MdAPE_%": (100 * e.groupby(bands, observed=True).median()).round(2),
                                  "MAPE_%": (100 * e.groupby(bands, observed=True).mean()).round(2)})
pd.concat(by_band, axis=1)

best feature set: kept_groups


base_ordinal        kept_groups       
                MdAPE_% MAPE_%     MdAPE_% MAPE_%
Q1 lowest          7.25  12.91        6.98  12.36
Q2                 5.88   9.25        5.72   8.84
Q3                 6.57   9.75        6.30   9.33
Q4                 8.71  11.57        8.47  11.30
Q5 highest        11.57  14.76       11.25  14.27

#### Step 7: What the best model relies on
Permutation importance on the last fold (Mar 2026), same idea as 04. Overlapping features share credit, so read groups, not single columns.

In [10]:
kind, num, cat, cols = keep_models[best]
train, val = folds[VAL_MONTHS[-1]]
model = make_model(kind, num, cat).fit(train[cols], train[TARGET].to_numpy())
mdape = make_scorer(lambda yt, yp: np.median(np.abs(yp - yt) / yt), greater_is_better=False)
pi = permutation_importance(model, val[cols], val[TARGET].to_numpy(), scoring=mdape, n_repeats=2, random_state=SEED)
importance = (pd.DataFrame({"MdAPE_increase_pts": 100 * pi.importances_mean}, index=cols)
                .sort_values("MdAPE_increase_pts", ascending=False).round(2))
importance.to_csv(os.path.join(MODEL_DIR, "fe_cv_permutation_importance.csv"))
importance.head(20)

,MdAPE_increase_pts
PostalCode,20.62
LivingArea,12.58
City,1.74
km_coast,0.87
geo_cluster,0.82
rot_45,0.79
km_job_center,0.77
MLSAreaMajor,0.47
SubdivisionName,0.41
fee_monthly,0.38
